# Advanced NLLB-200-1.3B Fine-Tuning with LoRA PEFT & Adaptive Length RAG

**Challenge:** Zindi Multilingual Health Question Answering in Low-Resource African Languages

**Model:** `facebook/nllb-200-1.3B` (1.3 Billion Parameters)

### Key Techniques in This Notebook:
1. **Bigger Model (`facebook/nllb-200-1.3B`)**: 1.3B parameter NLLB model offering superior translation and generation capacity for low-resource African languages.
2. **LoRA Parameter-Efficient Fine-Tuning (`--use_peft`)**: Reduces trainable parameters to < 1% (`r=16`, `lora_alpha=32`), enabling fast 1.3B training with low VRAM footprint.
3. **Tuned RAG Similarity Floor (`--min_similarity 0.25`)**: Context is injected into prompts ONLY when cosine similarity exceeds `0.25`, filtering out low-relevance noise.
4. **Language-Adaptive Length Bounds**: Enforces subset-specific bounds: Amharic (`Amh_Eth`: short ~20 words avg, `min_length=10, max_new_tokens=128`) vs Akan (`Aka_Gha`: long ~106 words avg, `min_length=35, max_new_tokens=320`).

In [ ]:
# 1. Environment & Repository Setup (Universal Home Root Reset)
import os, sys, shutil, zipfile, urllib.request
from pathlib import Path

# Reset working directory to home root
try:
    home_dir = Path.home()
    if (Path('/home/jovyan')).exists(): home_dir = Path('/home/jovyan')
    elif (Path('/content')).exists(): home_dir = Path('/content')
    os.chdir(home_dir)
except Exception as e:
    print(f'[WARN] Directory reset fallback: {e}')

repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'
zip_url = 'https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages/archive/refs/heads/main.zip'

if os.path.exists(repo_name):
    print(f"Cleaning previous '{repo_name}' directory...")
    shutil.rmtree(repo_name, ignore_errors=True)

if os.path.exists('repo.zip'):
    os.remove('repo.zip')

print('Downloading repository & raw datasets from GitHub...')
urllib.request.urlretrieve(zip_url, 'repo.zip')

print('Extracting project files...')
with zipfile.ZipFile('repo.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

if os.path.exists(f'{repo_name}-main'):
    os.rename(f'{repo_name}-main', repo_name)

%cd {repo_name}
print('\nSetup complete! Project root:')
!pwd


In [ ]:
# 2. Install Required Dependencies (Including PEFT & Sentence Transformers)
!pip install -q torch transformers datasets evaluate scikit-learn pandas numpy rouge-score sentence-transformers accelerate peft
print('All package dependencies installed successfully!')


In [ ]:
# 3. Self-Healing Path Setup & Master Imports
import os, sys, re, json, subprocess
from pathlib import Path

try:
    _cwd = Path.cwd()
except (FileNotFoundError, OSError):
    os.chdir('/home/jovyan')
    _cwd = Path.cwd()

repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'
if (_cwd / 'src').exists(): BASE_DIR = _cwd
elif (_cwd.parent / 'src').exists(): BASE_DIR = _cwd.parent
elif (_cwd / repo_name / 'src').exists(): BASE_DIR = _cwd / repo_name
elif (Path('/home/jovyan') / repo_name / 'src').exists(): BASE_DIR = Path('/home/jovyan') / repo_name
else: BASE_DIR = Path('/home/jovyan')

os.chdir(BASE_DIR)
for path_to_add in [str(BASE_DIR), str(BASE_DIR / 'src')]:
    if path_to_add not in sys.path: sys.path.insert(0, path_to_add)

DATA_DIR        = BASE_DIR / 'data' / 'raw'
SUBMISSIONS_DIR = BASE_DIR / 'submissions'
CHECKPOINTS_DIR = BASE_DIR / 'models' / 'checkpoints'
SRC_PATH        = BASE_DIR / 'src' / 'nllb_pipeline.py'

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'Training set.csv'
VAL_PATH   = DATA_DIR / 'Validation set.csv'
TEST_PATH  = DATA_DIR / 'Test set.csv'

import torch, pandas as pd, numpy as np
from retrieval import HybridRetriever
from threshold_optimizer import optimize_per_subset_thresholds

print(f'BASE_DIR: {BASE_DIR.resolve()}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Capacity: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
import subprocess
# 4. Fast Dry-Run Verification (NLLB-200-1.3B + LoRA PEFT + RAG Floor 0.25)
dry_run_cmd = [
    sys.executable, str(SRC_PATH),
    '--dry_run',
    '--model_name', 'facebook/nllb-200-1.3B',
    '--use_peft',
    '--use_dense_rag',
    '--min_similarity', '0.25',
    '--skip_submission',
]

print('Running 1.3B LoRA PEFT dry-run verification with live log streaming:')
process = subprocess.Popen(
    dry_run_cmd, cwd=str(BASE_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
assert process.returncode == 0, f'Dry run failed with return code {process.returncode}'


In [ ]:
import subprocess
# 5. Run Full NLLB-200-1.3B Fine-Tuning with LoRA PEFT & RAG Floor 0.25
full_run_cmd = [
    sys.executable, str(SRC_PATH),
    '--model_name', 'facebook/nllb-200-1.3B',
    '--use_peft',
    '--use_rag',
    '--use_dense_rag',
    '--min_similarity', '0.25',  # Tuned RAG floor filtering out noisy context
    '--epochs', '3',
    '--batch_size', '8',
    '--learning_rate', '2e-4',   # Higher LR for LoRA adapter tuning
    '--skip_submission',         # Evaluates validation ROUGE metrics first
    '--output_dir', str(CHECKPOINTS_DIR / 'nllb-1.3b-lora-checkpoint'),
]

print('🚀 Launching NLLB-200-1.3B LoRA PEFT Fine-Tuning & Per-Language Evaluation...')
process = subprocess.Popen(
    full_run_cmd, cwd=str(BASE_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
if process.returncode == 0:
    print('\n🎉 1.3B LoRA fine-tuning and validation evaluation complete!')
else:
    print(f'\n❌ Process exited with return code {process.returncode}')


In [ ]:
import subprocess
# 6. Evaluate Best Checkpoint & Optional Submission Generation
# Set GENERATE_SUBMISSION = True when validation metrics (especially Amharic vs Akan) are promising
GENERATE_SUBMISSION = True

sub_path = SUBMISSIONS_DIR / 'submission_nllb_1.3b_lora.csv'
ckpt_dir = CHECKPOINTS_DIR / 'nllb-1.3b-lora-checkpoint'

if GENERATE_SUBMISSION:
    print('Generating predictions for Test Set using NLLB-200-1.3B LoRA model...')
    sub_cmd = [
        sys.executable, str(SRC_PATH),
        '--model_name', 'facebook/nllb-200-1.3B',
        '--use_peft',
        '--use_rag',
        '--use_dense_rag',
        '--min_similarity', '0.25',
        '--submission_path', str(sub_path),
    ]
    process = subprocess.Popen(sub_cmd, cwd=str(BASE_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    if sub_path.exists():
        sub_df = pd.read_csv(sub_path)
        print(f'\n✅ 1.3B Submission Shape: {sub_df.shape}')
        display(sub_df.head(10))
else:
    print('GENERATE_SUBMISSION is False — Review validation ROUGE breakdown above. Set GENERATE_SUBMISSION = True to create submission CSV.')
